<a href="https://colab.research.google.com/github/Manikanta-23-ai/LANGCHAIN-LEARNING/blob/main/LangChain_Expression_Language.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install langchain_core

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 574.7/574.7 kB 7.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.6/883.6 kB 11.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 29.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8/8 [langchain_core]


In [3]:
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [4]:
prompt = ChatPromptTemplate.from_template("""
Answer the question using only the context provided.
If the answer is not present, say you could not find it in the PDF.

Context:
{context}

Question:
{question}

Answer:
""")

In [6]:
from google.colab import drive
drive.mount("/content/drive")

!pip install -q langchain-chroma langchain-huggingface sentence-transformers

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

DB_DIR = "/content/drive/MyDrive/langchain_project/chroma_db"

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

db = Chroma(
    collection_name="my_pdf",
    embedding_function=embedding_model,
    persist_directory=DB_DIR
)

retriever = db.as_retriever(
    search_kwargs={"k": 3}
)

print("Retriever loaded successfully!")

Mounted at /content/drive


Successfully renamed PrivateUse1 backend to 'tpu'. Device: device(type='tpu')
Registered Python module for 'tpu'.
Device type: tpu, Device index: default
Initializing TPU distributed runtime


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Retriever loaded successfully!


In [7]:
docs = retriever.invoke("What is the main topic of this PDF?")

for doc in docs:
    print("Page:", doc.metadata.get("page", "Unknown"))
    print(doc.page_content[:300])
    print("-" * 40)

Page: 141
Created By www.ebooktutorials.in
Content Downloaded from www.w3schools.com
142
----------------------------------------
Page: 151
Created By www.ebooktutorials.in
Content Downloaded from www.w3schools.com
152
----------------------------------------
Page: 38
Created By www.ebooktutorials.in
Content Downloaded from www.w3schools.com
39
----------------------------------------


In [12]:
!pip install -q transformers sentencepiece torch

import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

MODEL_NAME = "google/flan-t5-small"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

def llm(prompt, max_new_tokens=150, do_sample=False):
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(device)

    output = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=do_sample
    )

    answer = tokenizer.decode(
        output[0],
        skip_special_tokens=True
    )

    return [{"generated_text": answer}]

print("Model loaded successfully on", device)

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Model loaded successfully on cpu


In [13]:
print(
    rag_chain.invoke(
        "What is the main topic of this PDF?"
    )
)

ebooktutorials.in


In [14]:
from langchain_core.runnables import RunnableLambda

def prepare_inputs(question):
    docs = retriever.invoke(question)

    context = "\n\n".join(
        f"[Page {doc.metadata.get('page', 'Unknown')}]\n"
        f"{doc.page_content}"
        for doc in docs
    )

    return {
        "question": question,
        "context": context,
        "documents": docs
    }


def generate_answer_with_sources(inputs):
    full_prompt = prompt.invoke({
        "question": inputs["question"],
        "context": inputs["context"]
    }).to_string()

    response = llm(
        full_prompt,
        max_new_tokens=150,
        do_sample=False
    )

    return {
        "answer": response[0]["generated_text"],
        "sources": [
            {
                "page": doc.metadata.get("page", "Unknown"),
                "content": doc.page_content[:250]
            }
            for doc in inputs["documents"]
        ]
    }


rag_chain = (
    RunnableLambda(prepare_inputs)
    | RunnableLambda(generate_answer_with_sources)
)

In [15]:
result = rag_chain.invoke(
    "What is the main topic of this PDF?"
)

print("ANSWER:")
print(result["answer"])

print("\nSOURCE PAGES:")
for source in result["sources"]:
    print(f"\nPage: {source['page']}")
    print(source["content"])

ANSWER:
[Page 141]

SOURCE PAGES:

Page: 141
Created By www.ebooktutorials.in
Content Downloaded from www.w3schools.com
142

Page: 151
Created By www.ebooktutorials.in
Content Downloaded from www.w3schools.com
152

Page: 38
Created By www.ebooktutorials.in
Content Downloaded from www.w3schools.com
39


In [16]:
!pip install -q gradio

In [17]:
import gradio as gr

def pdf_chat(message, history):
    docs = retriever.invoke(message)

    if not docs:
        return "I couldn't find relevant content in the PDF."

    context = "\n\n".join(
        f"[Page {doc.metadata.get('page', 'Unknown')}]\n"
        f"{doc.page_content}"
        for doc in docs
    )

    prompt_text = f"""
Answer using only the provided PDF context.
If the answer is missing, say you could not find it.
Keep your answer clear and concise.

Context:
{context}

Question: {message}

Answer:
"""

    response = llm(
        prompt_text,
        max_new_tokens=150,
        do_sample=False
    )

    answer = response[0]["generated_text"]

    pages = sorted(set(
        str(doc.metadata.get("page", "Unknown"))
        for doc in docs
    ))

    return f"{answer}\n\nSource pages: {', '.join(pages)}"

In [18]:
demo = gr.ChatInterface(
    fn=pdf_chat,
    title="PDF AI Assistant",
    description="Ask questions about your PDF and inspect the source pages.",
    examples=[
        "What is this document about?",
        "Explain the key concepts.",
        "Summarize the main points."
    ]
)

demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://894df9642b21905b93.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
